# Навчання класифікатора «є корабель / немає»

Етап 5. Ноутбук проходить від початку до кінця і локально (CPU, `data/train_micro/` — перевірка логіки),
і на Kaggle (GPU — справжнє навчання). Перевірки `Dataset` — у `02_classifier.ipynb`.

**На Kaggle:**
1. Settings → Accelerator → **GPU**, Internet → **On**; секрет `KAGGLE_API_TOKEN` прикріплений;
   Input — `airbus-ship-detection-src` останньої версії.
2. Пробний запуск: `PILOT = True` (клітинка налаштувань у розділі 1) → **Run All** в інтерактивній сесії.
3. Повне навчання: `PILOT = False` → **Save Version → Save & Run All** (іде у фоні, вкладку можна закрити).
   Результати — у вкладці **Output** збереженої версії.

## 0. Дані на Kaggle

Локально цей розділ нічого не робить. На Kaggle: токен із секретів → завантаження архіву змагання (~5 хв)
→ розпакування лише `train_v2/` і CSV. Якщо дані вже розпаковані в цій сесії — крок пропускається.

In [ ]:
import os
from pathlib import Path

ON_KAGGLE = Path("/kaggle").exists()
KAGGLE_DATA_DIR = Path("/kaggle/tmp/data")

if ON_KAGGLE:
    from kaggle_secrets import UserSecretsClient
    os.environ["KAGGLE_API_TOKEN"] = UserSecretsClient().get_secret("KAGGLE_API_TOKEN")
    # Встановлена в Kaggle версія kaggle може не знати нового формату токена.
    !pip install -q kaggle==2.2.4

In [ ]:
if ON_KAGGLE and not (KAGGLE_DATA_DIR / "train_v2").exists():
    !mkdir -p /kaggle/tmp && cd /kaggle/tmp && kaggle competitions download -c airbus-ship-detection
    !unzip -q /kaggle/tmp/airbus-ship-detection.zip "train_v2/*" train_ship_segmentations_v2.csv -d {KAGGLE_DATA_DIR}

if ON_KAGGLE:
    n_files = sum(1 for _ in (KAGGLE_DATA_DIR / "train_v2").iterdir())
    print("знімків у train_v2:", n_files)
    assert n_files == 192556

## 1. Модель і цикл навчання

Той самий код працює локально і на Kaggle — відрізняються лише налаштування:

| | Локально (CPU) | Kaggle, `PILOT = True` | Kaggle, `PILOT = False` |
|---|---|---|---|
| знімки | `data/train_micro/` (300 шт.) | `/kaggle/tmp/data/train_v2/` | те саме |
| батчів навчання | 2 — перевірка логіки | 50 — замір швидкості | усі |
| val | 60 знімків з `train_micro` | 2 000 випадкових | усі 19 256 |
| епох | 1 | 1 | 4 |
| AMP (змішана точність) | вимкнено | увімкнено | увімкнено |
| результати | `data/classifier_local/` | `/kaggle/working/` | `/kaggle/working/` |

Pretrained-ваги ResNet34 (~85 МБ) завантажуються з інтернету при першому запуску
і далі беруться з кешу (`~/.cache/torch`).

In [ ]:
import random
import sys
import time

import numpy as np
import pandas as pd
import torch
import torchvision
from sklearn.metrics import roc_auc_score
from torch.utils.data import DataLoader

PILOT = True

if ON_KAGGLE:
    sys.path.append("/kaggle/input/datasets/yurapolishchuk/airbus-ship-detection-src")
    DATA_DIR = Path("/kaggle/tmp/data")
    IMAGES_DIR = DATA_DIR / "train_v2"
    OUT_DIR = Path("/kaggle/working")
    DEVICE = "cuda"
    MAX_TRAIN_BATCHES = 50 if PILOT else None
    MAX_VAL_IMAGES = 2000 if PILOT else None
    EPOCHS = 1 if PILOT else 4
    NUM_WORKERS = 4
else:
    ROOT = Path.cwd().parent
    sys.path.append(str(ROOT / "src"))
    DATA_DIR = ROOT / "data"
    IMAGES_DIR = DATA_DIR / "train_micro"
    OUT_DIR = DATA_DIR / "classifier_local"
    DEVICE = "cpu"
    MAX_TRAIN_BATCHES = 2
    MAX_VAL_IMAGES = None
    EPOCHS = 1
    NUM_WORKERS = 0

from dataset import ShipClassificationDataset, balanced_sample
from split import make_split

SEED = 42
BATCH_SIZE = 32 if ON_KAGGLE else 8
LR = 3e-4
USE_AMP = DEVICE == "cuda"
OUT_DIR.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
print("Kaggle:", ON_KAGGLE, "| PILOT:", PILOT, "| device:", DEVICE, "| AMP:", USE_AMP)

### 1.1. Дані

Поділ — тією самою функцією `make_split`, що створила `data/split.csv` (seed 42 → той самий поділ).
Локально `make_split` ділить лише 300 знімків `train_micro` (80/20), бо у справжню val їх потрапляє
надто мало.
Train — збалансований (`balanced_sample`), val — як є, з реальною часткою порожніх знімків.
У пілотному запуску val — 2 000 випадкових знімків, щоб замір не тривав довго.

In [ ]:
masks = pd.read_csv(DATA_DIR / "train_ship_segmentations_v2.csv")
if ON_KAGGLE:
    split = make_split(masks)
else:
    # З 300 знімків train_micro у справжню val потрапляє лише ~14, і всі з кораблями —
    # AUC не порахувати. Тому локально ділимо саму вибірку тією ж функцією.
    available = {p.name for p in IMAGES_DIR.iterdir()}
    split = make_split(masks[masks["ImageId"].isin(available)], val_size=0.2)

train_df = balanced_sample(split[split["split"] == "train"], seed=SEED)
val_df = split[split["split"] == "val"].reset_index(drop=True)
n_val_full = len(val_df)
if MAX_VAL_IMAGES is not None:
    val_df = val_df.sample(n=MAX_VAL_IMAGES, random_state=SEED).reset_index(drop=True)

train_dataset = ShipClassificationDataset(train_df["ImageId"], train_df["n_ships"] > 0, IMAGES_DIR, train=True)
val_dataset = ShipClassificationDataset(val_df["ImageId"], val_df["n_ships"] > 0, IMAGES_DIR, train=False)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS,
                          generator=torch.Generator().manual_seed(SEED))
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

print(f"train: {len(train_df)} знімків ({(train_df['n_ships'] > 0).mean():.0%} з кораблями)")
print(f"val:   {len(val_df)} знімків ({(val_df['n_ships'] > 0).mean():.0%} з кораблями)")

### 1.2. Модель

ResNet34 з вагами ImageNet. Останній шар `fc` (512 входів → 1000 класів ImageNet) замінюємо
на новий (512 → 1): один логіт «є корабель». Ваги нового шару випадкові — його вчимо з нуля,
решту мережі — донавчаємо (fine-tuning).

In [ ]:
model = torchvision.models.resnet34(weights="IMAGENET1K_V1")
model.fc = torch.nn.Linear(model.fc.in_features, 1)
model = model.to(DEVICE)

loss_fn = torch.nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
scaler = torch.amp.GradScaler(DEVICE, enabled=USE_AMP)

print(f"ваг у моделі: {sum(p.numel() for p in model.parameters()) / 1e6:.1f} млн")

### 1.3. Навчання однієї епохи і передбачення

`train_one_epoch` — 5 кроків на батч: forward → loss → `zero_grad` → `backward` → `step`.
З AMP `backward` і `step` йдуть через `scaler`, щоб малі градієнти не округлились до нуля.

`predict` — режим `eval()`, без градієнтів (`no_grad`), повертає ймовірності p після сигмоїди.

In [ ]:
def train_one_epoch(model, loader, max_batches=None):
    model.train()
    losses = []
    for i, (images, labels) in enumerate(loader):
        if max_batches is not None and i >= max_batches:
            break
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        with torch.autocast(device_type=DEVICE, enabled=USE_AMP):
            logits = model(images).squeeze(1)
            loss = loss_fn(logits, labels)
        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        losses.append(loss.item())
    return np.mean(losses)


@torch.no_grad()
def predict(model, loader):
    model.eval()
    probs, losses = [], []
    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        with torch.autocast(device_type=DEVICE, enabled=USE_AMP):
            logits = model(images).squeeze(1)
        losses.append(loss_fn(logits.float(), labels).item())
        probs.append(torch.sigmoid(logits.float()).cpu())
    return torch.cat(probs).numpy(), np.mean(losses)

### 1.4. Цикл по епохах

Після кожної епохи: train loss, val loss, val ROC-AUC, час. Якщо AUC кращий за всі попередні епохи —
зберігаємо ваги (`classifier_resnet34.pth`) і ймовірності для кожного val-знімка (`val_probs.csv`).

У пілотному запуску — ще й швидкість (знімків за секунду) і оцінка часу повної епохи.
Оцінка трохи завищена: перші батчі повільніші, поки запускаються процеси `DataLoader` і GPU «розігрівається».

In [ ]:
val_labels = (val_df["n_ships"] > 0).values
best_auc = -1
history = []

for epoch in range(1, EPOCHS + 1):
    start = time.time()
    train_loss = train_one_epoch(model, train_loader, MAX_TRAIN_BATCHES)
    train_seconds = time.time() - start
    val_probs, val_loss = predict(model, val_loader)
    val_seconds = time.time() - start - train_seconds
    auc = roc_auc_score(val_labels, val_probs)
    minutes = (time.time() - start) / 60
    history.append({"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss, "val_auc": auc, "minutes": minutes})
    print(f"епоха {epoch}: train loss {train_loss:.4f} | val loss {val_loss:.4f} | val AUC {auc:.4f} | {minutes:.1f} хв")

    if auc > best_auc:
        best_auc = auc
        torch.save(model.state_dict(), OUT_DIR / "classifier_resnet34.pth")
        val_df.assign(p=val_probs)[["ImageId", "n_ships", "p"]].to_csv(OUT_DIR / "val_probs.csv", index=False)
        print("    ↑ найкраща епоха — ваги і val_probs.csv збережено")

    if PILOT:
        n_train_seen = min(MAX_TRAIN_BATCHES or len(train_loader), len(train_loader)) * BATCH_SIZE
        train_speed = n_train_seen / train_seconds
        val_speed = len(val_df) / val_seconds
        full_epoch = len(train_df) / train_speed + n_val_full / val_speed
        print(f"    train: {train_speed:.0f} знімків/с, val: {val_speed:.0f} знімків/с")
        print(f"    оцінка повної епохи: {full_epoch / 60:.0f} хв "
              f"({len(train_df)} train + {n_val_full} val знімків)")

pd.DataFrame(history).to_csv(OUT_DIR / "history.csv", index=False)

### 1.5. Перевірка збережених результатів

- `val_probs.csv` — по рядку на кожен val-знімок, p від 0 до 1.
- Ваги завантажуються в нову модель, і вона дає ті самі передбачення.

In [ ]:
saved = pd.read_csv(OUT_DIR / "val_probs.csv")
print(saved.head())
assert len(saved) == len(val_df)
assert saved["p"].between(0, 1).all()

reloaded = torchvision.models.resnet34()
reloaded.fc = torch.nn.Linear(reloaded.fc.in_features, 1)
reloaded.load_state_dict(torch.load(OUT_DIR / "classifier_resnet34.pth", map_location=DEVICE))
reloaded_probs, _ = predict(reloaded.to(DEVICE), val_loader)
print("макс. різниця p після перезавантаження ваг:", np.abs(reloaded_probs - saved["p"].values).max())